# 11 • OpenAI + MCP: ค้นพบเครื่องมือสถานะคลัง

เป้าหมาย: ให้ MCP server ประกาศ tools แล้วใช้ LLM เลือก tool ส่งผลกลับและสรุปภาษาไทย

**ข้อมูลสมมติสำหรับการอบรม:** ชื่อหน่วยงาน Lot ผู้ขาย ค่าที่วัด และ กฏ ในตัวอย่างถูกสร้างขึ้นเพื่อเรียนรู้ AI ไม่ใช่ข้อมูลการผลิตหรือระเบียบจริงของกลุ่มศรีตรัง

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/biodatlab/sritrang-ai-workshop/blob/main/11_openai_mcp.ipynb)


## เตรียมพร้อมก่อนเริ่ม
แทนที่ `your_api_key` ในตัวแปร `OPENAI_API_KEY` ด้วย API key จริงของคุณ แล้วรัน cells จากบนลงล่าง ใช้รูปแบบเดียวกันทั้ง local และ Colab โดย cell ติดตั้ง packages อยู่ใน notebook แล้ว

ก่อนแชร์หรือ commit notebook ให้เปลี่ยน API key กลับเป็น `your_api_key`


In [ ]:
%pip install -q openai "mcp>=1.28,<2"

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = "your_api_key"  # แทนที่ด้วย API key จริงของคุณ
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4.1-mini"


## MCP server ที่เตรียมไว้
STDIO เปิด server เป็น process ภายใน Colab ไม่ต้องเปิด endpoint สาธารณะหรือเพิ่ม API key ใช้ SDK รุ่น v1 ที่กำหนดขอบเขตไว้


In [ ]:
%%writefile flood_server.py
from mcp.server.fastmcp import FastMCP
from typing import Literal

mcp = FastMCP("สถานะคลังสำหรับการอบรม")

@mcp.tool()
def get_flood_level(factory: Literal["คลังยางแท่ง A", "คลังน้ำยาง B"]):
    """อ่านระดับน้ำสมมติของคลังยางแท่ง A หรือคลังน้ำยาง B สำหรับการอบรม"""
    readings = {"คลังยางแท่ง A": 55, "คลังน้ำยาง B": 10}
    if factory not in readings:
        return {"error": "ไม่พบหน่วยงานนี้", "source": "ข้อมูลสมมติสำหรับการอบรม"}
    return {"factory": factory, "water_cm": readings[factory], "source": "ข้อมูลสมมติสำหรับการอบรม", "live": False}

if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
import json
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command=sys.executable, args=["flood_server.py"])

In [ ]:
async def mcp_exercise():
    with open("mcp_stderr.log", "w", encoding="utf-8") as errlog:
        async with stdio_client(server, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                discovered = await session.list_tools()
                tools = [{"type": "function", "name": t.name,
                          "description": t.description, "parameters": t.inputSchema}
                         for t in discovered.tools]
                print("Tools ที่ MCP ประกาศ:", [t["name"] for t in tools])
                print("Schema จาก MCP:", json.dumps(tools, ensure_ascii=False))
                messages = [{"role": "user", "content": "ตรวจระดับน้ำสมมติที่คลังยางแท่ง A แล้วสรุปเป็นภาษาไทย"}]
                response = client.responses.create(model=MODEL, input=messages, tools=tools,
                                                   tool_choice="required", max_output_tokens=500)
                messages += response.output
                for call in response.output:
                    if call.type == "function_call":
                        print("Tool ที่เลือก:", call.name, call.arguments)
                        result = await session.call_tool(call.name, json.loads(call.arguments))
                        output = result.model_dump_json()
                        print("ผลจาก MCP:", output)
                        messages.append({"type": "function_call_output", "call_id": call.call_id,
                                         "output": output})
                final = client.responses.create(model=MODEL, input=messages,
                    instructions="ตอบเป็นภาษาไทยและระบุชัดเจนว่าข้อมูลนี้เป็นข้อมูลสมมติสำหรับการอบรม", max_output_tokens=500)
                print(final.output_text)

await mcp_exercise()

## ลองทำด้วยตนเอง
1. เปลี่ยนคำถามเป็นคลังน้ำยาง B
2. เปรียบเทียบ schema ที่ค้นพบกับ notebook 10
3. อธิบายว่า MCP ทำให้ tool ใช้ซ้ำได้อย่างไร และใครเป็นผู้เลือก tool

ส่วนเสริมสำหรับวิทยากร: ใช้ HTTP MCP server ที่เข้าถึงได้ โดย native remote MCP ต้องมี endpoint ที่ OpenAI เข้าถึงได้
